Код тренировки

In [ ]:
import os
import random
import cv2 
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torch.optim import lr_scheduler

DEVICE = torch.device('cuda')


class ColorTripletDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.data_by_class = {}
        self.load_data()

    def load_data(self):
        for color_folder in os.listdir(self.root_dir):
            color_path = os.path.join(self.root_dir, color_folder)
            if os.path.isdir(color_path):
                images = [os.path.join(color_path, img) for img in os.listdir(color_path) if img.endswith('.jpg') or img.endswith('.png')]
                self.data_by_class[color_folder] = images

    def __len__(self):
        return sum(len(images) for images in self.data_by_class.values())

    def __getitem__(self, idx):
        # Случайно выбираем класс для anchor и positive
        positive_class = random.choice(list(self.data_by_class.keys()))
        positive_images = self.data_by_class[positive_class]

        # Выбираем anchor и positive из одного класса
        anchor_img_path = random.choice(positive_images)
        positive_img_path = random.choice(positive_images)
        
        while anchor_img_path == positive_img_path:
            positive_img_path = random.choice(positive_images)

        # Выбираем negative из другого класса
        negative_class = random.choice([key for key in self.data_by_class.keys() if key != positive_class])
        negative_img_path = random.choice(self.data_by_class[negative_class])

        # Чтение изображений
        anchor_img = cv2.imread(anchor_img_path)
        positive_img = cv2.imread(positive_img_path)
        negative_img = cv2.imread(negative_img_path)

        # Применение трансформаций
        if self.transform:
            anchor_img = self.transform(anchor_img)
            positive_img = self.transform(positive_img)
            negative_img = self.transform(negative_img)

        return anchor_img, positive_img, negative_img


# Простая сверточная сеть для извлечения признаков
class ColorNet(nn.Module):
    def __init__(self):
        super(ColorNet, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.fc1 = nn.Linear(128 * 8 * 8, 150)
    
    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.max_pool2d(x, 2)
        x = F.relu(self.conv2(x))
        x = F.max_pool2d(x, 2)
        x = F.relu(self.conv3(x))
        x = F.max_pool2d(x, 2)
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        return F.normalize(x)

# Triplet Loss
class TripletLoss(nn.Module):
    def __init__(self, margin=1.0):
        super(TripletLoss, self).__init__()
        self.margin = margin

    def forward(self, anchor, positive, negative):
        positive_distance = F.pairwise_distance(anchor, positive, p=2)
        negative_distance = F.pairwise_distance(anchor, negative, p=2)
        
        loss = torch.mean(F.relu(positive_distance - negative_distance + self.margin))
        return loss

def train_model(model, dataloader, loss_fn, optimizer, scheduler, num_epochs=10):
    model.train()
    for epoch in range(num_epochs):
        running_loss = 0.0
        for img_anchor, img_positive, img_negative in dataloader:
            img_anchor = img_anchor.to(DEVICE)
            img_positive = img_positive.to(DEVICE)
            img_negative = img_negative.to(DEVICE)

            output_anchor = model(img_anchor)
            output_positive = model(img_positive)
            output_negative = model(img_negative)

            loss = loss_fn(output_anchor, output_positive, output_negative)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
        
        scheduler.step()
        
        avg_loss = running_loss / len(dataloader)
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}")

root_dir = '../../data/player_classification'

transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((64, 64)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
])

# Создание датасета и DataLoader
dataset = ColorTripletDataset(root_dir, transform=transform)
dataloader = DataLoader(dataset, batch_size=40, shuffle=True)

model = ColorNet().to(DEVICE)
loss_fn = TripletLoss(margin=1.0)
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = lr_scheduler.StepLR(optimizer, step_size=4, gamma=0.7)

# Обучение модели
train_model(model, dataloader, loss_fn, optimizer, scheduler, num_epochs=24)

# torch.save(model.state_dict(), 'model.pth')

In [ ]:
# Перевод в onnx

import torch
import torch.onnx

model.eval()
dummy_input = torch.randn(1, 3, 64, 64).to(DEVICE)

onnx_path = '../../weights/player_classification.onnx'
torch.onnx.export(
    model,                    # Модель для экспорта
    dummy_input,              # Пример ввода для модели
    onnx_path,                # Путь для сохранения модели
    export_params=True,       # Сохранение параметров модели
    opset_version=11,         # Версия ONNX opset
    do_constant_folding=True, # Оптимизация констант
    input_names=['input'],    # Имена входных тензоров
    output_names=['output'],  # Имена выходных тензоров
    dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}}  # Поддержка динамического размера батча
)

Inference

In [ ]:
import onnxruntime as ort
import numpy as np
import cv2
from PIL import Image
from torchvision import transforms
import matplotlib.pyplot as plt

onnx_model_path = '../../weights/player_classification.onnx'
ort_session = ort.InferenceSession(onnx_model_path)

transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
])

def extract_features_batch(image_paths, transform):
    embeddings = []
    
    for image_path in image_paths:
        image = cv2.imread(image_path)
        image = Image.fromarray(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
        image = transform(image)
        
        image = image.unsqueeze(0).numpy()

        ort_inputs = {ort_session.get_inputs()[0].name: image}
        embedding = ort_session.run(None, ort_inputs)[0]
        embeddings.append(embedding.squeeze())
    
    return np.stack(embeddings)

def find_most_different(embeddings):
    n = embeddings.shape[0]
    max_distance = 0
    index1, index2 = 0, 0

    for i in range(n):
        for j in range(i + 1, n):
            distance = 1 - np.dot(embeddings[i], embeddings[j]) / (np.linalg.norm(embeddings[i]) * np.linalg.norm(embeddings[j]))
            if distance > max_distance:
                max_distance = distance
                index1, index2 = i, j
    
    return index1, index2

root1 = 'red_cska'
root2 = 'green_dark'

image_paths = [f'data/{root1}/{x}' for x in os.listdir(f'data/{root1}')] + [f'data/{root2}/{x}' for x in os.listdir(f'data/{root2}')]
image_paths = [x for x in image_paths if not '.DS_Store' in x]

embeddings = extract_features_batch(image_paths, transform)
idx1, idx2 = find_most_different(embeddings)

img1 = cv2.imread(image_paths[idx1])
img2 = cv2.imread(image_paths[idx2])

plt.imshow(cv2.cvtColor(img1, cv2.COLOR_BGR2RGB))
plt.show()
plt.imshow(cv2.cvtColor(img2, cv2.COLOR_BGR2RGB))
plt.show()

In [ ]:
anchor1 = embeddings[idx1]
anchor2 = embeddings[idx2]

labels = []
for i, emb in enumerate(embeddings):
    if i == idx1 or i == idx2:
        labels.append(-1)
        continue
    
    dist_to_class1 = np.linalg.norm(emb - anchor1)
    dist_to_class2 = np.linalg.norm(emb - anchor2)

    img = cv2.imread(image_paths[i])
    print(dist_to_class1, dist_to_class2)
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.show()

    if dist_to_class1 < dist_to_class2:
        labels.append(1)
    else:
        labels.append(2)

for i, label in enumerate(labels):
    if label != -1:  # Пропускаем эталонные изображения
        print(f"Image {image_paths[i]} assigned to class {label}")